# Sketch to App: Building with Generative AI

Welcome! In this workshop you will **draw an app on paper or an iPad, and have AI build a working version of it**.

Along the way you'll learn:
1. What generative AI is and where it goes wrong
2. How to write prompts that get you what you want
3. How to check AI's work instead of trusting it blindly
4. How to put your app online for free

**No coding experience needed.** You'll run each cell by clicking the ▶ button on its left (or pressing `Shift + Enter`). Read the text, change the parts marked `✏️`, and run.

| Part | What you'll do |
| --- | --- |
| 0 | Set up (5 min) |
| 1 | Intro to AI |
| 2 | Brainstorm an app idea with AI |
| 3 | Sketch it and have AI build it |
| 4 | Make it better with clearer prompts |
| 5 | Check the AI's work |
| 6 | Download and publish your app |
| 7 | Wrap-up |

---
## Part 0: Setup

This notebook uses **Google Gemini**, which has a free API key.

**Get your key (2 min):**
1. Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey) and sign in with a Google account.
2. Click **Create API key** and copy it.
3. In this notebook, click the **🔑 key icon** in the left sidebar (Secrets).
4. Click **Add new secret**. Name it `GOOGLE_API_KEY` and paste your key as the value.
5. Turn on **Notebook access** for that secret.

> 🔒 Never paste your key directly into a cell or share it. Secrets keep it private.

Then run the cell below.

In [ ]:
# Install the Gemini library and connect with your key
!pip install -q -U google-genai

import io, re
from google import genai
from google.colab import userdata, files
from IPython.display import HTML, Markdown, display
from PIL import Image

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"  # a fast, free-tier Gemini model


def ask(prompt):
    """Send a text prompt to Gemini and show the answer."""
    response = client.models.generate_content(model=MODEL, contents=prompt)
    display(Markdown(response.text))
    return response.text


def extract_html(text):
    """Pull the HTML code out of the AI's reply."""
    match = re.search(r"```(?:html)?\s*(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()


def show_app(html):
    """Run the app right here in the notebook."""
    display(HTML(f'<div style="border:2px solid #ccc;border-radius:8px;padding:8px">{html}</div>'))


ask("Say hello to a workshop full of new AI users in one short, friendly sentence.")

✅ If you see a greeting above, you're connected!

**Stuck?** Check that your secret is named exactly `GOOGLE_API_KEY` and has Notebook access turned on. If you can't get a key, you can still follow along by pasting the prompts into [Gemini](https://gemini.google.com/), [ChatGPT](https://chatgpt.com/), or [Claude](https://claude.ai/).

---
## Part 1: What is generative AI?

**Generative AI** creates new things (text, images, code) instead of just searching for them.

Tools like ChatGPT, Gemini and Claude run on **large language models (LLMs)**. An LLM learned patterns from huge amounts of text and works by **predicting the most likely next word**, over and over.

That means it:
- ✅ is great at drafting, explaining, brainstorming, summarizing and **writing code**
- ⚠️ sounds confident **whether or not it's right** (this is called a *hallucination*)
- ⚠️ only knows what's in your conversation, so **the more detail you give, the better it does**
- ⚠️ can reflect biases in what it learned from

> **The big idea for today:** AI is a fast, eager assistant. You're still the boss. Give clear instructions and check its work.

Try it: run the cell below, then change the question and run it again.

In [ ]:
# ✏️ Change the question to anything you're curious about
ask("In 3 short bullet points, explain how an AI chatbot works to a 12-year-old.")

---
## Part 2: Brainstorm an app idea

Not sure what to build? Let AI help you brainstorm. Fill in the blanks below and run it.

In [ ]:
# ✏️ Fill these in
who_i_am = "a college student"
something_i_care_about = "staying organized and drinking enough water"

ask(f"""I'm {who_i_am} and I care about {something_i_care_about}.
Give me 6 ideas for a simple one-screen app I could sketch on paper.
Keep each idea to one sentence. Mix practical ideas with fun ones.""")

Pick one idea (or your own). Keep it **small**: one or two screens is perfect for today.

---
## Part 3: Sketch it and have AI build it

### Step 1: Sketch (about 5 min)
On paper or your iPad, draw the **main screen** of your app.

### Step 2: Label everything
Next to each button, box, or section, write **what it does**. For example:
- *"+ button: opens a box to type a new task"*
- *"Checkbox: crosses the task out when tapped"*
- *"Top bar: shows the app name and today's date"*

### Step 3: Take a photo
Snap a clear photo of your sketch (or export it from your iPad), then run the cell below and upload it.

In [ ]:
# Upload a photo of your sketch
uploaded = files.upload()
filename = next(iter(uploaded))
sketch = Image.open(io.BytesIO(uploaded[filename]))
sketch.thumbnail((800, 800))
display(sketch)

### Step 4: Describe your app
Type out what each part of your sketch does. **The more specific, the better.** Then run the cell to have AI build it.

In [ ]:
# ✏️ Describe your app. One line per button or section.
my_description = """
App name: Water Buddy
- Top: app name and a big number showing glasses of water today
- "+1 Glass" button: adds one to the number
- A progress bar that fills up toward a goal of 8 glasses
- "Reset" button: sets the count back to 0
"""

build_prompt = f"""Build the app in this sketch as a single, complete HTML file
(HTML, CSS and JavaScript all in one file, no external files).

Every button must actually work as described here:
{my_description}

Make it clean, colorful, and work well on a phone screen.
Reply with only the HTML code."""

response = client.models.generate_content(model=MODEL, contents=[sketch, build_prompt])
app_v1 = extract_html(response.text)
show_app(app_v1)

🎉 **Try out your app above!** Click the buttons. Does it work the way you drew it?

Write down:
- One thing the AI got **right**
- One thing it got **wrong or skipped**

---
## Part 4: Make it better with clearer prompts

The first try is rarely perfect. Here's the most important skill in using AI:

### A good prompt has 5 parts

| Part | Ask yourself | Example |
| --- | --- | --- |
| **Role** | Who should the AI act as? | "You are a friendly app designer." |
| **Task** | What exactly do I want? | "Build a water-tracking app." |
| **Context** | Who is it for, and why? | "For busy students who forget to drink water." |
| **Format** | What should the result look like? | "One HTML file, big buttons, blue and white colors." |
| **Constraints** | Limits or must-haves? | "No sign-in. Must work offline." |

### See the difference
Run this cell. It sends the **same sketch** with a lazy prompt. Compare it to your app from Part 3.

In [ ]:
# A weak, vague prompt, for comparison
weak = client.models.generate_content(
    model=MODEL,
    contents=[sketch, "make this into an app. html only."],
)
show_app(extract_html(weak.text))

Which one is closer to your sketch? Usually the detailed one wins, because **the AI can only follow instructions you actually give it.**

### Now improve your own app
Ask for changes in plain English. Run this cell as many times as you like, changing the request each time.

In [ ]:
# ✏️ Ask for one change at a time
change_request = "Make the buttons bigger and add a fun message when I reach my goal."

current_app = app_v1 if "current_app" not in globals() else current_app

improve_prompt = f"""Here is my app's HTML code:

{current_app}

Change it like this: {change_request}
Keep everything else working. Reply with the full updated HTML file only."""

response = client.models.generate_content(model=MODEL, contents=[sketch, improve_prompt])
current_app = extract_html(response.text)
show_app(current_app)

💡 **Prompting tips**
- **One change at a time** works better than five at once.
- **Be specific:** "make the title bigger and dark blue" beats "make it look better."
- **Ask it to ask you:** "Before building, ask me 3 questions about what I want."
- **Say what's broken:** "When I click Reset, nothing happens. Fix it."

---
## Part 5: Check the AI's work

AI makes mistakes **confidently**. Before you trust what it made, check it.

### Compare three AI tools
1. Run the cell below to print your prompt.
2. Open [ChatGPT](https://chatgpt.com/), [Gemini](https://gemini.google.com/), and [Claude](https://claude.ai/) in new tabs.
3. In each one, upload your sketch photo and paste the prompt.
4. Most of these tools show a preview of the app right in the chat. Try each one.

In [ ]:
print(build_prompt)

### Score each result (1 = poor, 3 = great)

Double-click this cell to fill in your scores.

| Question | ChatGPT | Gemini | Claude |
| --- | --- | --- | --- |
| Does it **look like my sketch**? | | | |
| Do all the **buttons work** as I labeled them? | | | |
| Did it **add things I didn't ask for**? (that's guessing) | | | |
| Did it **skip** anything? | | | |
| Is it **easy to use**? | | | |
| **Total** | | | |

### 🚩 Red flags to watch for in any AI answer
- Buttons that look right but **don't do anything**
- Features it **says** it added but didn't
- Very specific facts, numbers, or sources with no way to check them
- A confident tone on something it can't actually know

---
## Part 6: Download and publish your app

### Download it
Run the cell to save your app as `index.html`. Double-click the downloaded file to open it in any browser, on any computer.

In [ ]:
final_app = current_app if "current_app" in globals() else app_v1
with open("index.html", "w") as f:
    f.write(final_app)
files.download("index.html")

### Put it online for free with GitHub Pages (optional)
1. Sign in or sign up at [github.com](https://github.com).
2. Click **+** (top right) → **New repository**. Name it something like `water-buddy`, set it to **Public**, and click **Create repository**.
3. Click **uploading an existing file**, drag in your `index.html`, and click **Commit changes**.
4. Go to **Settings** → **Pages**. Under *Branch*, pick **main** and click **Save**.
5. Wait about a minute. Your app is live at `https://YOUR-USERNAME.github.io/water-buddy/` 🎉

Share the link with a friend and have them try it on their phone!

---
## Part 7: Wrap-up

### Five habits for using AI well
1. **Be specific.** Role, task, context, format, constraints.
2. **Iterate.** Treat the first answer as a rough draft.
3. **Verify.** Test the code, check the facts, compare tools.
4. **Protect privacy.** Never paste passwords, personal info, or private data.
5. **Be honest.** Follow your school or work rules about AI, and say when you used it.

### Talk it over
- What surprised you about what the AI built?
- Where would you **not** want to use AI?

### ✏️ Before you go
Write one prompt you'll actually use this week, using the 5 parts. Double-click here to type it.

> *My prompt:*

**Mahalo for coming!** 🌺